# 03 Modeling Win Probability

Train a simple baseline model on synthetic data and score active deals for guidance.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from customer_pricing_analytics.model_training import train_baseline_logistic_regression
from customer_pricing_analytics.scoring import score_active_deals

rng = np.random.default_rng(7)
n = 160
df = pd.DataFrame({
    'product_type': rng.choice(['RCF', 'Term Loan', 'Guarantee'], n),
    'rating': rng.choice(['A', 'BBB', 'BB'], n),
    'tenor_months': rng.choice([12, 24, 36, 60], n),
    'margin_above_floor_bps': rng.normal(40, 22, n),
    'total_expected_profit': rng.normal(50000, 15000, n).clip(5000),
})
linear_score = -0.2 + 0.012 * df['margin_above_floor_bps'] - 0.35 * (df['rating'] == 'BB') + 0.2 * (df['product_type'] == 'RCF')
probability = 1 / (1 + np.exp(-linear_score))
df['deal_outcome'] = np.where(rng.random(n) < probability, 'won', 'lost')

feature_cols = ['product_type', 'rating', 'tenor_months', 'margin_above_floor_bps', 'total_expected_profit']
model, metrics = train_baseline_logistic_regression(df, 'deal_outcome', feature_cols)
metrics

In [ ]:
active_deals = df[feature_cols].sample(8, random_state=1).reset_index(drop=True)
score_active_deals(model, active_deals)

The score is a decision-support probability. It should be calibrated and governed before production use, and it should not be interpreted as an automatically optimal price.